In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
import os



In [2]:
# Load splits
train_data = pd.read_pickle('dataset/train_data_final.pkl')
val_data_masked = pd.read_pickle('dataset/val_data_masked.pkl')
test_data_masked = pd.read_pickle('dataset/test_data_masked.pkl')

In [3]:
# Load ground truth
val_ground_truth = pd.read_pickle('dataset/val_ground_truth.pkl')
test_ground_truth = pd.read_pickle('dataset/test_ground_truth.pkl')

# Load mask indicators
val_mask_indicator = pd.read_pickle('dataset/val_mask_indices.pkl')
test_mask_indicator = pd.read_pickle('dataset/test_mask_indices.pkl')





In [4]:
print(f"\nLoaded all data:")
print(f"    Train: {train_data.shape}")
print(f"    Val (masked): {val_data_masked.shape}")
print(f"    Test (masked): {test_data_masked.shape}")


Loaded all data:
    Train: (143459, 75)
    Val (masked): (30741, 75)
    Test (masked): (30742, 75)


In [5]:
# Get features that were masked
features_to_impute = val_ground_truth.columns.tolist()
print(f"\n  Features to impute: {len(features_to_impute)}")


  Features to impute: 37


In [6]:
# Get all columns
all_columns = train_data.columns.tolist()

# Identify non-impute features
non_impute_features = [col for col in all_columns if col not in features_to_impute]

print(f"\n FEATURE SPLIT:")
print(f"    Features to impute: {len(features_to_impute)}")
print(f"    Features to keep as-is: {len(non_impute_features)}")


 FEATURE SPLIT:
    Features to impute: 37
    Features to keep as-is: 38


In [7]:
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler

In [ ]:

def clip_outliers_per_feature(data, lower_pct=0.5, upper_pct=99.5):
    """
    Clip outliers per feature based on percentiles.
    Handles NaN values.
    """
    data_clipped = data.copy()
    
    for i in range(data.shape[1]):
        col = data[:, i]
        
        # Get valid (non-NaN) values
        valid_mask = ~np.isnan(col)
        
        if valid_mask.sum() > 0:
            valid_values = col[valid_mask]
            
            # Calculate percentiles
            lower_bound = np.percentile(valid_values, lower_pct)
            upper_bound = np.percentile(valid_values, upper_pct)
            
            # Clip only valid values
            col[valid_mask] = np.clip(valid_values, lower_bound, upper_bound)
            data_clipped[:, i] = col
    
    return data_clipped

In [9]:

train_values = train_data[features_to_impute].values

In [10]:
train_values_clipped = clip_outliers_per_feature(train_values, lower_pct=0.5, upper_pct=99.5)


In [11]:
for i, feat in enumerate(features_to_impute[:10]):  # Show first 10
    original_max = np.nanmax(np.abs(train_values[:, i]))
    clipped_max = np.nanmax(np.abs(train_values_clipped[:, i]))
    
    if original_max > 1000:  # Only show features that were clipped significantly
        print(f"  • {feat[:45]:45s}: {original_max:>12.2e} → {clipped_max:>12.2e}")

In [12]:
scaler = StandardScaler()
scaler.fit(train_values_clipped)

# Transform all datasets (with outlier clipping)
train_scaled = scaler.transform(train_values_clipped)

val_values_clipped = clip_outliers_per_feature(val_data_masked[features_to_impute].values)
val_scaled = scaler.transform(val_values_clipped)

test_values_clipped = clip_outliers_per_feature(test_data_masked[features_to_impute].values)
test_scaled = scaler.transform(test_values_clipped)

In [14]:

new_stats = {
    'Min': np.nanmin(train_scaled),
    'Max': np.nanmax(train_scaled),
    'Mean': np.nanmean(train_scaled),
    'Std': np.nanstd(train_scaled),
    '1st percentile': np.nanpercentile(train_scaled, 1),
    '99th percentile': np.nanpercentile(train_scaled, 99)
}

print(f"{'Statistic':<20s} {'Value':<15s}")
print("─"*40)
for stat, value in new_stats.items():
    print(f"{stat:<20s} {value:<15.4f}")


# Check max values per feature
print(f"\nTop 5 features by max scaled value:")
feature_max_scaled = []
for i, feat in enumerate(features_to_impute):
    max_val = np.nanmax(np.abs(train_scaled[:, i]))
    feature_max_scaled.append((feat, max_val))

feature_max_scaled.sort(key=lambda x: x[1], reverse=True)

for feat, max_val in feature_max_scaled[:5]:
    print(f"  {feat[:45]:45s}: {max_val:>10.4f}")


Statistic            Value          
────────────────────────────────────────
Min                  -3.9694        
Max                  8.8233         
Mean                 0.0000         
Std                  1.0000         
1st percentile       -2.0953        
99th percentile      2.2409         

Top 5 features by max scaled value:
  jitter                                       :     8.8233
  Pos in Ref Round                             :     3.9694
  PCell_Uplink_TB_Size                         :     3.9400
  Altitude                                     :     3.8259
  Traffic Jam Factor                           :     3.3163


In [15]:
import numpy as np
import pandas as pd
from sklearn.experimental import enable_iterative_imputer 
from sklearn.impute import KNNImputer, IterativeImputer, SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [16]:

def evaluate_imputation(imputed_data, ground_truth, mask_indicator, method_name, split_name):
  
    results = []
    all_errors_squared = []
    all_errors_abs = []
    
    for col in ground_truth.columns:
        # Get mask for this feature
        mask = mask_indicator[col]
        
        # Count masked values
        n_masked = mask.sum()
        
        if n_masked == 0:
            continue
        
        # Get ground truth values (only non-NaN)
        true_vals = ground_truth.loc[mask, col].dropna()
        
        
        # Get predicted values at same positions
        pred_vals = imputed_data.loc[true_vals.index, col]
        
        # Calculate errors
        errors = true_vals - pred_vals
        errors_squared = errors ** 2
        errors_abs = errors.abs()
        
        rmse = np.sqrt(errors_squared.mean())
        mae = errors_abs.mean()
        
        # Store results
        results.append({
            'feature': col,
            'n_masked': len(true_vals),
            'RMSE': rmse,
            'MAE': mae
        })
        
        # Collect for overall metrics
        all_errors_squared.extend(errors_squared.values)
        all_errors_abs.extend(errors_abs.values)
    
    # Overall metrics
    overall_rmse = np.sqrt(np.mean(all_errors_squared))
    overall_mae = np.mean(all_errors_abs)
    
    results_df = pd.DataFrame(results)
    
    return results_df, overall_rmse, overall_mae

print("\n EVALUATING METHODS...")



 EVALUATING METHODS...


In [20]:
def represent_columnwise_results(results_df, split_name="", method_name="", sort_by="RMSE", ascending=False, top_n=None):
    """
    """
    if results_df is None or results_df.empty:
        print("No column-wise results to display.")
        return results_df

    view = results_df.copy()

    # Ensure numeric formatting columns exist
    for c in ["n_masked", "RMSE", "MAE"]:
        if c in view.columns:
            view[c] = pd.to_numeric(view[c], errors="coerce")

    if sort_by in view.columns:
        view = view.sort_values(sort_by, ascending=ascending)

    if top_n is not None:
        view = view.head(top_n)

    title = f"\nColumn-wise Results"
    if split_name:
        title += f" | Split: {split_name}"
    if method_name:
        title += f" | Method: {method_name}"
    print(title)
    print("-" * len(title))

    print(
        view.to_string(
            index=False,
            formatters={
                "RMSE": lambda x: f"{x:.4f}" if pd.notna(x) else "nan",
                "MAE": lambda x: f"{x:.4f}" if pd.notna(x) else "nan",
            }
        )
    )

    return view


In [ ]:

# Create KNN imputer
knn_imputer = KNNImputer(n_neighbors=5, weights='uniform')


knn_imputer.fit(train_scaled)

# Transform validation (on scaled data)
print("  Imputing validation set...")
val_knn_scaled = knn_imputer.transform(val_scaled)

# Inverse transform back to original scale
val_knn_original = scaler.inverse_transform(val_knn_scaled)

# Create full dataframe
val_knn = val_data_masked.copy()
val_knn[features_to_impute] = val_knn_original

# Transform test (on scaled data)
print("  Imputing test set...")
test_knn_scaled = knn_imputer.transform(test_scaled)

# Inverse transform back to original scale
test_knn_original = scaler.inverse_transform(test_knn_scaled)

# Create full dataframe
test_knn = test_data_masked.copy()
test_knn[features_to_impute] = test_knn_original


  Imputing validation set...
  Imputing test set...


In [26]:

method_name = 'KNN Imputation'


# Validation
val_results, val_rmse, val_mae = evaluate_imputation(
    val_knn, val_ground_truth, val_mask_indicator,
    method_name, 'Validation'
)

# Test
test_results, test_rmse, test_mae = evaluate_imputation(
    test_knn, test_ground_truth, test_mask_indicator,
    method_name, 'Test'
)

print(f"\n  VALIDATION:")
print(f"      Overall RMSE: {val_rmse:.4f}")
print(f"      Overall MAE:  {val_mae:.4f}")
print(f"      Features evaluated: {len(val_results)}")

print(f"\n  TEST:")
print(f"      Overall RMSE: {test_rmse:.4f}")
print(f"      Overall MAE:  {test_mae:.4f}")
print(f"      Features evaluated: {len(test_results)}")
    


  VALIDATION:
      Overall RMSE: 39827.8522
      Overall MAE:  142.2052
      Features evaluated: 39

  TEST:
      Overall RMSE: 22720.5874
      Overall MAE:  94.4895
      Features evaluated: 39
